# Chapitre 10 · Le Transformer : assemble ton GPT (solutions des exercices)

Ce notebook contient **uniquement les réponses aux sept exercices** du notebook du
chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

La première cellule reprend le minimum de la leçon (vocabulaire, hyperparamètres,
modèles) pour que chaque validation s'exécute de façon autonome. Le GPT y est **non
entraîné** : c'est suffisant pour valider les exercices.

In [1]:
# Mise en place (reprise de la leçon) : le minimum pour que les validations tournent.
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# Le vocabulaire exact des trente fables (81 caractères), sans réembarquer le corpus.
chars = list('\n !"\'(),-.:;?ABCDEFGHIJLMNOPQRSTUVXYabcdefghijlmnopqrstuvxyzÀÂÇÈÉÊÔÛàâçèéêîïôùûŒœ')
vocab_size = len(chars)                          # 81
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.randint(0, vocab_size, (2000,))     # séquence factice : seules les shapes comptent ici

block_size = 64
d_model, n_heads, n_layers, d_ff = 96, 4, 2, 384


def fabriquer_batch(taille=32):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y


class MultiHeadAttention(nn.Module):
    """La version de la leçon, reprise compacte."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model doit être divisible par n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, dm = x.shape
        Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask[:T, :T] == 0, float("-inf"))
        poids = torch.softmax(scores, dim=-1)
        out = (poids @ V).transpose(1, 2).contiguous().view(B, T, dm)
        return self.W_O(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.table_tokens = nn.Embedding(vocab_size, d_model)
        self.table_positions = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)
        self.register_buffer("masque", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T = x.shape
        h = self.table_tokens(x) + self.table_positions(torch.arange(T))
        for bloc in self.blocs:
            h = bloc(h, self.masque)
        return self.tete(self.ln_final(h))


gpt = GPT()    # NON entraîné : suffisant pour valider les exercices
print("Mise en place OK :", vocab_size, "caractères de vocabulaire |",
      sum(p.numel() for p in gpt.parameters()), "paramètres.")


Mise en place OK : 81 caractères de vocabulaire | 244800 paramètres.


### Exercice 1 · Le découpage en têtes — niveau ●

`(B, T, d_model)` devient `(B, n_heads, T, d_k)`. Deux gestes : `view(B, T, n_heads, d_k)`
(on renomme le découpage, rien ne bouge en mémoire), puis `transpose(1, 2)` (l'axe des
têtes passe devant, pour que PyTorch calcule les `B × n_heads` attentions comme un seul
gros batch). Et souviens-toi du cas qui échoue : le `view` direct a la bonne shape et
les mauvaises valeurs.

In [2]:
def decouper_en_tetes(X, n_heads):
    """(B, T, d_model) -> (B, n_heads, T, d_k), avec d_k = d_model // n_heads.

    La tête h du token t doit recevoir les colonnes [h*d_k : (h+1)*d_k] du vecteur du token t.
    """
    B, T, d_model = X.shape
    d_k = d_model // n_heads
    X = X.view(B, T, n_heads, d_k).transpose(1, 2)
    return X

In [3]:
X = torch.arange(2 * 3 * 8, dtype=torch.float32).view(2, 3, 8)   # valeurs 0..47, faciles à suivre
H = decouper_en_tetes(X, n_heads=2)
assert isinstance(H, torch.Tensor), "remplace le ... par ton code"
assert H.shape == (2, 2, 3, 4), f"shape {tuple(H.shape)} au lieu de (2, 2, 3, 4) : view puis transpose(1, 2)"
assert torch.equal(H[0, 0, 1], X[0, 1, :4]), "la tête 0 doit recevoir les 4 premières colonnes de chaque token"
assert torch.equal(H[0, 1, 2], X[0, 2, 4:]), "la tête 1 doit recevoir les 4 dernières colonnes de chaque token"
print("Exercice 1 validé : chaque tête reçoit sa tranche de chaque token.")

Exercice 1 validé : chaque tête reçoit sa tranche de chaque token.


### Exercice 2 · Le masque top-k — niveau ●

À chaque pas de génération, top-k ne garde que les `k` tokens les plus probables et met
tous les autres à `-inf` (probabilité exactement nulle après softmax). Refais le geste
central sur un vecteur jouet : garde les 3 meilleurs scores, coupe le reste. Le seuil est
`v[:, [-1]]`, le k-ième score ; une seule ligne avec `masked_fill`, et la shape ne doit
pas changer.

In [4]:
logits_jouet = torch.tensor([[5.0, 4.0, 3.0, 2.0, 1.0, 0.0]])
v, _ = torch.topk(logits_jouet, 3)               # les 3 plus grands scores

logits_coupes = logits_jouet.masked_fill(logits_jouet < v[:, [-1]], float("-inf"))


In [5]:
# Validation : le masque top-k.
attendu = torch.tensor([[5.0, 4.0, 3.0, float("-inf"), float("-inf"), float("-inf")]])
assert isinstance(logits_coupes, torch.Tensor), "remplace le ... par ton code"
assert logits_coupes.shape == logits_jouet.shape, "le masque ne doit PAS changer la shape"
assert torch.equal(logits_coupes, attendu), "top-k doit garder les 3 meilleurs, couper le reste à -inf"
print("Exercice 2 validé : les recalés valent -inf, le softmax leur donnera zéro exactement.")


Exercice 2 validé : les recalés valent -inf, le softmax leur donnera zéro exactement.


### Exercice 3 · Le comptage de paramètres — niveau ●●

Le chapitre te l'a promis : le comptage des 244 800 paramètres, tu le refais toi-même,
à la main. Rappels : `nn.Linear(a, b, bias=False)` pèse `a × b` nombres ; avec biais,
`a × b + b` ; `nn.Embedding(n, d)` pèse `n × d` ; `nn.LayerNorm(d)` pèse `2 × d`
(gamma et bêta). Nos hyperparamètres : `vocab_size = 81`, `block_size = 64`,
`d_model = 96`, `d_ff = 384`, `n_layers = 2`.

In [6]:
nb_attention = 4 * d_model * d_model                                  # 36 864
nb_ffn = (d_model * d_ff + d_ff) + (d_ff * d_model + d_model)         # 74 208
nb_bloc = nb_attention + nb_ffn + 2 * (2 * d_model)                   # 111 456
nb_total = (vocab_size * d_model + block_size * d_model               # les deux tables
            + n_layers * nb_bloc                                      # les blocs
            + 2 * d_model                                             # LayerNorm final
            + d_model * vocab_size)                                   # la tête de sortie


In [7]:
# Validation : le comptage, étage par étage, puis contre le modèle réel.
assert nb_attention == 36_864, f"attention : {nb_attention} au lieu de 36 864 (4 matrices d_model x d_model)"
assert nb_ffn == 74_208, f"FFN : {nb_ffn} au lieu de 74 208 (n'oublie pas les 2 biais)"
assert nb_bloc == 111_456, f"bloc : {nb_bloc} au lieu de 111 456 (attention + FFN + 2 LayerNorm)"
assert nb_total == 244_800, f"total : {nb_total} au lieu de 244 800"
assert nb_total == sum(p.numel() for p in GPT().parameters()), "ton compte doit coller au modèle réel"
print(f"Exercice 3 validé : {nb_total} paramètres, dont {n_layers * nb_ffn} dans les FFN.")


Exercice 3 validé : 244800 paramètres, dont 148416 dans les FFN.


### Exercice 4 · Le bloc Transformer — niveau ●●

Deux sous-couches, et pour chacune le même emballage : une LayerNorm AVANT (Pre-LN),
et une connexion résiduelle qui AJOUTE la sortie à l'entrée au lieu de la remplacer :

```
x = x + attention(LayerNorm(x))
x = x + ffn(LayerNorm(x))
```

Le `x +` est l'autoroute du gradient. La validation inclut le « test du résidu »,
qui attrape un câblage Post-LN.

In [8]:
class TransformerBlock(nn.Module):
    """L'unité qu'on empile : attention + FFN, chacun avec Pre-LN et résidu."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)   # collecter (et garder l'autoroute)
        x = x + self.ffn(self.ln2(x))          # digérer (idem)
        return x

In [9]:
torch.manual_seed(0)
bloc = TransformerBlock(d_model=96, n_heads=4, d_ff=384)
x_test = torch.randn(2, 10, 96)
masque_test = torch.tril(torch.ones(10, 10))

y_test = bloc(x_test, masque_test)
assert isinstance(y_test, torch.Tensor), "remplace le ... par ton code"
assert y_test.shape == x_test.shape, "un bloc préserve la shape : c'est ce qui permet d'empiler"
assert not torch.allclose(y_test, x_test), "le bloc doit transformer x, pas le renvoyer tel quel"

# Le test du résidu : on éteint les deux sous-couches (dernières projections à zéro).
# Si ton câblage est bien x + souscouche(LN(x)), le bloc doit laisser passer x INTACT.
# (En Post-LN, x = LN(x + souscouche(x)), il sortirait LN(x), pas x.)
with torch.no_grad():
    bloc.attn.W_O.weight.zero_()
    bloc.ffn.net[2].weight.zero_()
    bloc.ffn.net[2].bias.zero_()
assert torch.allclose(bloc(x_test, masque_test), x_test, atol=1e-6), (
    "sous-couches éteintes, le bloc doit rendre x intact : vérifie tes résidus et le Pre-LN"
)
print("Exercice 4 validé : ton bloc transforme, préserve la shape, et l'autoroute résiduelle est en place.")

Exercice 4 validé : ton bloc transforme, préserve la shape, et l'autoroute résiduelle est en place.


### Exercice 5 · Le GPT complet — niveau ●●

Trois étages dans le `forward` : les embeddings (tokens + positions, additionnés), la
pile de blocs (chacun avec `self.masque`), puis la tête de sortie
(`self.tete(self.ln_final(h))`). La validation vérifie le nombre de paramètres, la
shape de sortie, et surtout la causalité : le passé ne doit pas voir le futur.

In [10]:
class GPT(nn.Module):
    """Embeddings + positions -> n_layers blocs -> LayerNorm -> tête de sortie."""

    def __init__(self):
        super().__init__()
        self.table_tokens = nn.Embedding(vocab_size, d_model)
        self.table_positions = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)
        self.register_buffer("masque", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T = x.shape
        h = self.table_tokens(x) + self.table_positions(torch.arange(T))   # (B, T, d_model)
        for bloc in self.blocs:
            h = bloc(h, self.masque)                                       # shape préservée
        return self.tete(self.ln_final(h))                                 # (B, T, vocab_size)

In [11]:
torch.manual_seed(42)
gpt_ex = GPT()
n_params_ex = sum(p.numel() for p in gpt_ex.parameters())
assert n_params_ex == 244_800, f"{n_params_ex} paramètres au lieu de 244 800 : relis les hyperparamètres"

x_ex, _ = fabriquer_batch(taille=4)
logits_ex = gpt_ex(x_ex)
assert logits_ex is not None and isinstance(logits_ex, torch.Tensor), "remplace le ... par ton code"
assert logits_ex.shape == (4, block_size, vocab_size), f"shape {tuple(logits_ex.shape)} au lieu de (4, 64, 81)"

# Le test de causalité : changer le DERNIER caractère ne doit rien changer aux
# prédictions des positions précédentes (le passé ne voit pas le futur).
x2_ex = x_ex.clone()
x2_ex[:, -1] = (x2_ex[:, -1] + 1) % vocab_size
logits2_ex = gpt_ex(x2_ex)
assert torch.allclose(logits_ex[:, :-1], logits2_ex[:, :-1], atol=1e-5), (
    "changer le futur change le passé : le masque causal n'est pas branché"
)
assert not torch.allclose(logits_ex[:, -1], logits2_ex[:, -1]), "la dernière position devrait, elle, changer"
print("Exercice 5 validé : ton GPT sort un score par caractère et par position, et respecte la causalité.")


Exercice 5 validé : ton GPT sort un score par caractère et par position, et respecte la causalité.


### Exercice 6 · La génération — niveau ●●

La même mécanique qu'au chapitre 1 : prédire des probabilités, tirer au sort, décaler
le contexte, recommencer. Seule différence : le GPT rend un score par position, et
seul le DERNIER pas nous intéresse pour prédire la suite (`logits[:, -1, :]`).
Quatre lignes dans la boucle.

In [12]:
def generer(model, prompt="\n", longueur=300, temperature=0.8):
    """Écrit `longueur` caractères à la suite de `prompt`, avec le GPT."""
    model.eval()
    ctx = ([stoi["\n"]] * block_size + [stoi[c] for c in prompt])[-block_size:]
    sortie = []
    with torch.no_grad():
        for _ in range(longueur):
            logits = model(torch.tensor([ctx]))[:, -1, :]            # le dernier pas seulement
            probas = torch.softmax(logits / temperature, dim=-1)     # le softmax du chapitre 5
            i = torch.multinomial(probas, num_samples=1).item()      # tirage au sort
            sortie.append(itos[i])
            ctx = ctx[1:] + [i]                                      # on décale le contexte
    return prompt + "".join(sortie)

In [13]:
torch.manual_seed(7)
texte = generer(gpt, prompt="La cigale", longueur=80)
assert isinstance(texte, str) and texte.startswith("La cigale"), "le texte doit commencer par le prompt"
assert len(texte) == len("La cigale") + 80, f"longueur {len(texte)} au lieu de {len('La cigale') + 80}"
assert set(texte) <= set(chars), "tous les caractères générés doivent venir du vocabulaire"
print("Exercice 6 validé : ton GPT écrit.")

Exercice 6 validé : ton GPT écrit.


### Exercice 7 · La classe `MultiHeadAttention` — niveau ●●●

Le morceau de bravoure : tout le chapitre 9, plus le découpage en têtes. Le forward,
en cinq gestes :

1. projeter `x` en Q, K, V (trois `nn.Linear`), puis découper chacun en têtes ;
2. scores : `Q @ K.transpose(-2, -1) / math.sqrt(d_k)` ;
3. masque causal (s'il est fourni) : `-inf` sur le futur, AVANT le softmax ;
4. softmax ligne par ligne, puis moyenne pondérée des values ;
5. recoller les têtes (`transpose(1, 2).contiguous().view(B, T, d_model)`) et projeter par `W_O`.

La validation compare ta sortie à `nn.MultiheadAttention`, l'implémentation
officielle de PyTorch, à 10⁻⁵ près.

In [14]:
class MultiHeadAttention(nn.Module):
    """n_heads attentions en parallèle dans des sous-espaces de dimension d_k."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model doit être divisible par n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)   # mixe les têtes recollées

    def forward(self, x, mask=None):
        B, T, d_model = x.shape
        Q = decouper_en_tetes(self.W_Q(x), self.n_heads)      # (B, h, T, d_k)
        K = decouper_en_tetes(self.W_K(x), self.n_heads)
        V = decouper_en_tetes(self.W_V(x), self.n_heads)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)   # (B, h, T, T)
        if mask is not None:
            scores = scores.masked_fill(mask[:T, :T] == 0, float('-inf'))
        poids = torch.softmax(scores, dim=-1)
        melange = poids @ V                                   # (B, h, T, d_k)
        out = melange.transpose(1, 2).contiguous().view(B, T, d_model)
        return self.W_O(out)

In [15]:
# Validation contre l'implémentation officielle de PyTorch : on copie les poids
# de nn.MultiheadAttention dans ta classe, et les sorties doivent coïncider.
torch.manual_seed(0)
d_test, h_test, T_test = 32, 4, 7

reference = nn.MultiheadAttention(d_test, h_test, bias=False, batch_first=True)
mienne = MultiHeadAttention(d_test, h_test)
with torch.no_grad():
    Wq, Wk, Wv = reference.in_proj_weight.chunk(3, dim=0)
    mienne.W_Q.weight.copy_(Wq)
    mienne.W_K.weight.copy_(Wk)
    mienne.W_V.weight.copy_(Wv)
    mienne.W_O.weight.copy_(reference.out_proj.weight)

x_test = torch.randn(2, T_test, d_test)
interdit = torch.triu(torch.ones(T_test, T_test, dtype=torch.bool), diagonal=1)
sortie_ref, _ = reference(x_test, x_test, x_test, attn_mask=interdit, need_weights=False)
sortie_mienne = mienne(x_test, mask=torch.tril(torch.ones(T_test, T_test)))

assert sortie_mienne is not None and isinstance(sortie_mienne, torch.Tensor), "remplace le ... par ton code"
assert sortie_mienne.shape == (2, T_test, d_test), f"shape {tuple(sortie_mienne.shape)}"
assert torch.allclose(sortie_ref, sortie_mienne, atol=1e-5), (
    "ta sortie diffère de nn.MultiheadAttention : vérifie le découpage en têtes, "
    "le masque (avant le softmax !) et le recollage"
)
print("Exercice 7 validé : ta MultiHeadAttention colle à nn.MultiheadAttention de PyTorch.")


Exercice 7 validé : ta MultiHeadAttention colle à nn.MultiheadAttention de PyTorch.
